In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier

In [2]:
train_part1 = pd.read_parquet("../data/test.parquet", engine='fastparquet')
event_id = train_part1["event_id"]

In [3]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 633683 entries, 0 to 633682
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   customer_id                 633683 non-null  int64  
 1   event_id                    633683 non-null  int64  
 2   event_dttm                  633683 non-null  object 
 3   event_type_nm               633683 non-null  int32  
 4   event_desc                  633683 non-null  int32  
 5   channel_indicator_type      633683 non-null  int32  
 6   channel_indicator_sub_type  633683 non-null  int32  
 7   operaton_amt                293202 non-null  float64
 8   currency_iso_cd             301466 non-null  float64
 9   mcc_code                    154227 non-null  object 
 10  pos_cd                      42205 non-null   float64
 11  accept_language             63501 non-null   object 
 12  browser_language            53907 non-null   object 
 13  timezone                 

In [4]:
delete = ["accept_language", "browser_language"]

train_part1.drop(columns=delete, inplace=True)

In [5]:
#########
# columns = ["event_type_nm", "channel_indicator_type", "channel_indicator_sub_type", "currency_iso_cd", "mcc_code", "pos_cd", "timezone",\
#            "developer_tools", "phone_voip_call_state", "web_rdp_connection", "compromised", "operating_system_type", "event_desc"]
# for i in columns:
#     train_part1[i] = train_part1[i].fillna(value=-1, inplace=False)
#     train_part1[i] = train_part1[i].astype(dtype="int16")

In [6]:
train_part1["Hour"] = pd.to_datetime(train_part1["event_dttm"]).dt.hour.astype(dtype="int16")
train_part1.drop(columns="event_dttm", inplace=True)

In [7]:
train_part1.drop(columns=["customer_id", "event_id"], inplace=True)

In [8]:
# columns = ["event_type_nm", "channel_indicator_type", "channel_indicator_sub_type", "currency_iso_cd", "mcc_code", "pos_cd", "timezone",\
#            "developer_tools", "phone_voip_call_state", "web_rdp_connection", "compromised", "operating_system_type", "event_desc"]

# for i in columns:
#     # train_part1 = train_part1.with_columns(pl.when(pl.col(i) == -1).then(None).otherwise(pl.col(i)).alias(i))
#     train_part1[i] = train_part1[i].fillna('missing').astype(str)

In [9]:
cat_features = [
    'mcc_code', 'event_desc',
    'timezone', 'operating_system_type', 'device_system_version',
    'screen_size', 'battery'
]

for i in cat_features:
    # train_part1 = train_part1.with_columns(pl.col(i).fill_null('missing'))
    train_part1[i] = train_part1[i].fillna('missing').astype(str)

In [10]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 633683 entries, 0 to 633682
Data columns (total 19 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   event_type_nm               633683 non-null  int32  
 1   event_desc                  633683 non-null  str    
 2   channel_indicator_type      633683 non-null  int32  
 3   channel_indicator_sub_type  633683 non-null  int32  
 4   operaton_amt                293202 non-null  float64
 5   currency_iso_cd             301466 non-null  float64
 6   mcc_code                    633683 non-null  str    
 7   pos_cd                      42205 non-null   float64
 8   timezone                    633683 non-null  str    
 9   session_id                  420464 non-null  float64
 10  operating_system_type       633683 non-null  str    
 11  battery                     633683 non-null  str    
 12  device_system_version       633683 non-null  str    
 13  screen_size              

Получили обработанный набор данных, теперь можно переходить к обучению моделей.

In [11]:
model = CatBoostClassifier()
model.load_model('../Models/Model_4.cbm')

CatBoostClassifier(class_names=[0, 1], class_weights=[1, 100], depth=5, iterations=50, loss_function='Logloss', verbose=0)

In [12]:
predict = model.predict(train_part1)
y_pred_proba = model.predict_proba(train_part1)[:, 1]

In [13]:
y_pred_proba = pd.Series(y_pred_proba, name="predict")
y_pred_proba

0         0.000005
1         0.000004
2         0.000002
3         0.000002
4         0.000003
            ...   
633678    0.000004
633679    0.000002
633680    0.000002
633681    0.000002
633682    0.000002
Name: predict, Length: 633683, dtype: float64

In [14]:
event_id

0         123707242230467
1         123234793229123
2         125837545055907
3         126456020999239
4         125090221587312
               ...       
633678    125528306953183
633679    123690063072824
633680    125734465101827
633681    125193299828684
633682    125657156303168
Name: event_id, Length: 633683, dtype: int64

In [15]:
submit = pd.concat([event_id, y_pred_proba], names=["event_id", "predict"], axis=1)

In [16]:
submit

,event_id,predict
0,123707242230467,0.000005
1,123234793229123,0.000004
2,125837545055907,0.000002
3,126456020999239,0.000002
4,125090221587312,0.000003
...,...,...
633678,125528306953183,0.000004
633679,123690063072824,0.000002
633680,125734465101827,0.000002
633681,125193299828684,0.000002


In [17]:
submit.to_csv("submit.csv", index=False)